# 04 — Video Processor
Builds the final 1080x1920 short with image motion, subtitles, optional music, then a blank source card. No self/host video.

In [ ]:
import os,sys,subprocess
ROOT="/content/bhf"
REPO_URL=""  # put your GitHub repo URL here if you want automatic git clone
if not os.path.exists(ROOT):
    if not REPO_URL: raise RuntimeError("Set REPO_URL to your GitHub repository before running.")
    subprocess.run(["git","clone",REPO_URL,ROOT],check=True)
sys.path.insert(0,ROOT)
from factory.drive import mount_drive,DrivePaths
from factory.config import Config
MYDRIVE=mount_drive(); paths=DrivePaths(os.path.join(MYDRIVE,"BLACK_HISTORY_FACTORY")); paths.ensure_tree(); config=Config.load(paths.root)
print(paths.root)


In [ ]:
from factory.video_engine import run
from factory.utils import read_json,write_json_atomic
from factory import status
while True:
    jobs=[j for j in os.listdir(paths("02_JOBS")) if (read_json(paths.manifest(j),{}) or {}).get("status")=="AUDIO_READY"]
    if not jobs: print("No video jobs available."); break
    for job_id in jobs:
        try:
            scenes=read_json(paths.scenes(job_id),[]); research=read_json(paths.research(job_id),{}); status.set_processor(paths,"video","running",job_id,"rendering")
            final=run(paths,job_id,scenes,research,config)
            d=read_json(paths.manifest(job_id),{}); d.update(status="COMPLETED",final_video=final); write_json_atomic(paths.manifest(job_id),d)
            status.set_processor(paths,"video","idle",job_id,final); print("COMPLETED",job_id,final)
        except Exception as e:
            d=read_json(paths.manifest(job_id),{}); d.update(status="VIDEO_ERROR",video_error=str(e)); write_json_atomic(paths.manifest(job_id),d); status.set_processor(paths,"video","error",job_id,str(e))
